# LCA Lasers4MaaS: workflow assembly

Adds up the material, pre-joining, joining and post-joining impacts of each product, for each joining technology.

Run `joining_task.ipynb` and `pre_post.ipynb` first. This notebook reads their result files in `results/`.
The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [28]:
import pandas as pd
from IPython.display import display

from l4m_lca import plots
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.constants import EU27_GB, JOINING_STEM, MATERIAL_STEM, PREPOST_STEM
from l4m_lca.costs import COST_COLUMNS, COST_UNITS
from l4m_lca.project import (DEFAULT_INDICATORS, ef31_methods, find_activity, method_units, open_project,
                             select_background)
from l4m_lca.workflow import (STAGE_TITLES, cost_table, cumulative_regional, material_table, read_results, unit_of,
                              workflow_table, write_json)

## Parameters

In [29]:
# Background database: "ecoinvent" or "bafu". The environment variable L4M_BACKGROUND overrides it.
BG = select_background("ecoinvent")
BACKGROUND = BG.name

# Result files of the other notebooks
JOINING_RESULTS = f"{BG.results}/{JOINING_STEM}_LCIA_results.json"
PREPOST_RESULTS = f"{BG.results}/{PREPOST_STEM}_LCIA_results.json"
REGIONAL_JOINING = f"{BG.results}/regional_joining.csv"
REGIONAL_PREPOST = f"{BG.results}/regional_prepost.csv"
MATERIAL_RESULTS = f"{BG.results}/{MATERIAL_STEM}_LCIA_results.json"

INDICATORS = DEFAULT_INDICATORS
DEFAULT_IMPACT = "climate change: total (EF v3.1 - IPCC 2021)"
SCENARIO = "amount"           # "minimum", "amount" or "maximum"
MATERIAL_CASE = "prototype"   # "prototype" or "production": selects the part weight
INCLUDE_MATERIAL = False     # True adds the material production stage to the workflow results

# Material markets (name, location) of each background; keys = PRODUCT_WORKFLOWS[...]["material"]
MATERIAL_MARKETS = {
    "ecoinvent": {"aluminium": ("market for aluminium, wrought alloy", "GLO"),
                  "steel": ("market for steel, chromium steel 18/8", "GLO")},
    "bafu": {"aluminium": ("Aluminium, production mix, wrought alloy, at plant", "RER"),
             "steel": ("Chromium steel 18/8, at plant", "RER")},
}

In [30]:
# Product-specific workflow: material, enabled pre- and post-joining steps, prototype/production parameters.
# Joint lengths are in joining_task.ipynb (the reference for the joining step).
PRODUCT_WORKFLOWS = {
    "Heat exchanger": {
        "material": "steel",          # AISI 316 stainless steel (use-case specifications)
        "pre_joining": {"alkaline_degreasing": True, "mechanical_treatment": True, "machining": False},
        "post_joining": {"pickling": True, "passivation": True, "grinding_polishing": True},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 0.5, "max": 2.0},
                "part_weight_kg": {"min": 1.0, "max": 5.0},
            },
            "production": {
                "surface_area_m2": {"min": 1.0, "max": 5.0},
                "part_weight_kg": {"min": 5.0, "max": 20.0},
            },
        },
    },
    "Fuel tank": {
        "material": "aluminium",      # Al 6061 (use-case specifications)
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,  # oxide layer removal before joining
            "machining": True,             # fuel tanks require tight tolerances (no aluminium machining data yet: counts 0)
        },
        "post_joining": {"chemical_etching": True, "conversion_coating": True, "grinding_polishing": True},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 2.0, "max": 10.0},
                "part_weight_kg": {"min": 10.0, "max": 50.0},
            },
            "production": {
                "surface_area_m2": {"min": 5.0, "max": 20.0},
                "part_weight_kg": {"min": 50.0, "max": 200.0},
            },
        },
    },
    "Heterogeneous pipe": {
        "material": "steel",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,  # oxides/scale present on pipes
            "machining": False,
        },
        "post_joining": {"pickling": True, "passivation": True, "grinding_polishing": True},
        "process_parameters": {   # no part weight yet: the material stage is 0
            "pipe_diameter_mm": {"min": 50, "max": 200},
            "pipe_length_m": {"min": 1.0, "max": 10.0},
        },
    },
    "Freeze Dryers Shelf": {
        "material": "steel",
        "pre_joining": {"alkaline_degreasing": True, "mechanical_treatment": True, "machining": False},
        "post_joining": {"pickling": True, "passivation": True, "grinding_polishing": False},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 1.0, "max": 3.0},
                "part_weight_kg": {"min": 20.0, "max": 100.0},
            },
            "production": {
                "surface_area_m2": {"min": 3.0, "max": 10.0},
                "part_weight_kg": {"min": 100.0, "max": 300.0},
            },
        },
    },
    "Battery cooling plate": {
        "material": "aluminium",
        "pre_joining": {"alkaline_degreasing": True, "mechanical_treatment": True, "machining": False},
        "post_joining": {"chemical_etching": True, "conversion_coating": False, "grinding_polishing": True},
        "process_parameters": {
            "prototype": {
                "surface_area_m2": {"min": 0.1, "max": 0.5},
                "part_weight_kg": {"min": 1.26, "max": 14.38},
            },
            "production": {
                "surface_area_m2": {"min": 0.5, "max": 2.0},
                "part_weight_kg": {"min": 14.38, "max": 57.52},
            },
        },
    },
}

## Material production LCIA (per kg)

EF 3.1 impacts of 1 kg of each material market. The material stage of a product = impact per kg x mean part weight of `MATERIAL_CASE`.

In [31]:
open_project(BG.project, BG.database)
methods = ef31_methods(INDICATORS)
units = method_units(methods)
markets = {m: find_activity(BG.database, *spec) for m, spec in MATERIAL_MARKETS[BACKGROUND].items()}
impacts = BackgroundImpacts(methods, next(iter(markets.values())))

material_df = material_table(markets, impacts, units, BG.database)
write_json(material_df, MATERIAL_RESULTS)
material_df.set_index("Key").T

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)


Key,aluminium,steel
Database,ecoinvent-3.12-cutoff,ecoinvent-3.12-cutoff
Process,"market for aluminium, wrought alloy [GLO]","market for steel, chromium steel 18/8 [GLO]"
Scenario,amount,amount
Functional unit,1 kilogram,1 kilogram
acidification [mol H+-Eq],0.084753,0.028299
eutrophication: terrestrial [mol N-Eq],0.142728,0.054828
human toxicity: carcinogenic [CTUh],0.0,0.0
"human toxicity: carcinogenic, inorganics [CTUh]",0.0,0.0
"human toxicity: carcinogenic, organics [CTUh]",0.0,0.0
particulate matter formation [disease incidence],0.000001,0.0


## Workflow assembly

In [32]:
joining_df = read_results(JOINING_RESULTS)
prepost_df = read_results(PREPOST_RESULTS)
workflow_df = workflow_table(
    PRODUCT_WORKFLOWS, joining_df, prepost_df, material_df, MATERIAL_CASE, SCENARIO,
    include_material=INCLUDE_MATERIAL,
)

workflow_df[workflow_df["indicator"].str.startswith(DEFAULT_IMPACT)].pivot_table(
    index=["product", "technology"],
    columns=["stage", "step"],
    values="value",
    sort=False,
)

stage                                                               pre  \
step                                                alkaline_degreasing   
product               technology                                          
Heat exchanger        standard laser                           0.162395   
                      L4M-solution                             0.162395   
Fuel tank             L4M-solution                             0.198565   
                      TIG                                      0.198565   
Heterogeneous pipe    standard laser                           0.162395   
                      L4M-solution                             0.162395   
Freeze Dryers Shelf   standard laser                           0.162395   
                      L4M-solution                             0.162395   
Battery cooling plate L4M-solution                             0.198565   
                      controlled atmosphere brazing            0.198565   

stage                                                                     \
step                                                mechanical_treatment   
product               technology                                           
Heat exchanger        standard laser                            0.136109   
                      L4M-solution                              0.136109   
Fuel tank             L4M-solution                              0.073153   
                      TIG                                       0.073153   
Heterogeneous pipe    standard laser                            0.136109   
                      L4M-solution                              0.136109   
Freeze Dryers Shelf   standard laser                            0.136109   
                      L4M-solution                              0.136109   
Battery cooling plate L4M-solution                              0.073153   
                      controlled atmosphere brazing             0.073153   

stage                                                   post              \
step                                                pickling passivation   
product               technology                                           
Heat exchanger        standard laser                 0.40835    1.050288   
                      L4M-solution                   0.40835    1.050288   
Fuel tank             L4M-solution                       NaN         NaN   
                      TIG                                NaN         NaN   
Heterogeneous pipe    standard laser                 0.40835    1.050288   
                      L4M-solution                   0.40835    1.050288   
Freeze Dryers Shelf   standard laser                 0.40835    1.050288   
                      L4M-solution                   0.40835    1.050288   
Battery cooling plate L4M-solution                       NaN         NaN   
                      controlled atmosphere brazing      NaN         NaN   

stage                                                                   \
step                                                grinding_polishing   
product               technology                                         
Heat exchanger        standard laser                          0.242692   
                      L4M-solution                            0.242692   
Fuel tank             L4M-solution                            0.108876   
                      TIG                                     0.108876   
Heterogeneous pipe    standard laser                          0.242692   
                      L4M-solution                            0.242692   
Freeze Dryers Shelf   standard laser                               NaN   
                      L4M-solution                                 NaN   
Battery cooling plate L4M-solution                            0.108876   
                      controlled atmosphere brazing           0.108876   

stage                                                     join       pre  \
step   

In [33]:
display(plots.workflow_bars(workflow_df, STAGE_TITLES, DEFAULT_IMPACT, SCENARIO))

## Operating costs

Consumable costs, personnel costs and nominal OPEX (consumable + personnel) of each step, in EUR per product, in the `SCENARIO` scenario.
Personnel costs use the EU27 average labour cost. Labour hours: joining time x operators (joining_task.ipynb), and process time x operator share for the pre- and post-joining steps (pre_post.ipynb: 0.25 for chemical baths, 1 for mechanical treatments). The material stage has no costs.
The bar chart above also shows these indicators (select them in the indicator list).

In [34]:
cost_table(workflow_df, COST_COLUMNS).round(3)

indicator                                           Consumable costs (EUR)  \
stage                                                                  pre   
product               technology                                             
Heat exchanger        standard laser                                 0.203   
                      L4M-solution                                   0.203   
Fuel tank             L4M-solution                                   0.171   
                      TIG                                            0.171   
Heterogeneous pipe    standard laser                                 0.203   
                      L4M-solution                                   0.203   
Freeze Dryers Shelf   standard laser                                 0.203   
                      L4M-solution                                   0.203   
Battery cooling plate L4M-solution                                   0.171   
                      controlled atmosphere brazing                  0.171   

indicator                                                                   \
stage                                                 post    join   total   
product               technology                                             
Heat exchanger        standard laser                 0.861  56.074  57.137   
                      L4M-solution                   0.861  28.952  30.015   
Fuel tank             L4M-solution                   0.141   2.962   3.274   
                      TIG                            0.141   7.386   7.699   
Heterogeneous pipe    standard laser                 0.861   0.032   1.096   
                      L4M-solution                   0.861   0.018   1.081   
Freeze Dryers Shelf   standard laser                 0.735  14.625  15.563   
                      L4M-solution                   0.735   0.622   1.560   
Battery cooling plate L4M-solution                   0.141   0.151   0.463   
                      controlled atmosphere brazing  0.141  33.224  33.536   

indicator                                           Personnel costs (EUR)  \
stage                                                                 pre   
product               technology                                            
Heat exchanger        standard laser                                8.750   
                      L4M-solution                                  8.750   
Fuel tank             L4M-solution                                  7.292   
                      TIG                                           7.292   
Heterogeneous pipe    standard laser                                8.750   
                      L4M-solution                                  8.750   
Freeze Dryers Shelf   standard laser                                8.750   
                      L4M-solution                                  8.750   
Battery cooling plate L4M-solution                                  7.292   
                      controlled atmosphere brazing                 7.292   

indicator                                                                    \
stage                                                  post    join   total   
product               technology                                              
Heat exchanger        standard laser                 15.312  32.274  56.337   
                      L4M-solution                   15.312  16.683  40.746   
Fuel tank             L4M-solution                    6.125   1.968  15.384   
                      TIG                             6.125  19.678  33.094   
Heterogeneous pipe    standard laser                 15.312   0.049  24.111   
                      L4M-solution                   15.312   0.037  24.099   
Freeze Dryers Shelf   standard laser                  6.562  14.259  29.572   
                      L4M-solution                    6.562   1.283  16.596   
Battery cooling plate L4M-solution                    6.125   0.668  14.085   
                    

## Cumulative regional analysis: EU and UK electricity markets

Adds up the regional impacts and operating costs of the enabled pre-joining steps, the joining process and the enabled post-joining steps of each product, with the electricity market and the labour cost of each EU member state and of the United Kingdom (GB).
Countries without a Eurostat labour cost (for example NL, BE) have empty personnel costs and nominal OPEX.
The material stage is not included. Pre/post steps and joining use the `SCENARIO` scenario.

In [35]:
reg_joining = pd.read_csv(REGIONAL_JOINING)
reg_prepost = pd.read_csv(REGIONAL_PREPOST)
cumulative_df = cumulative_regional(PRODUCT_WORKFLOWS, reg_joining, reg_prepost, SCENARIO)

regional_units = {**units, **COST_UNITS}
for col in joining_df.columns:   # units from the result column names, e.g. "acidification [mol H+-Eq]"
    if "[" in col:
        regional_units.setdefault(col.split(" [")[0], unit_of(col))

display(plots.regional_explorer(
    cumulative_df,
    filters=[("Product", "product", list(PRODUCT_WORKFLOWS), None),
             ("Technology", "technology", sorted(cumulative_df["technology"].unique()), None)],
    units=regional_units,
    title=lambda f: f"{f['product']}, {f['technology']}: cumulative pre + joining + post",
    countries=EU27_GB, scatter_on_top=True, height=1400, width=1400,
))

## GHG emissions of the joining stage and production costs per use case

Top: climate change of the joining stage of each product (LCA, `SCENARIO` scenario). Bottom: total production costs per item, per cost type, from `Input_data/KPI_calculations.xlsx` (see `production_costs.ipynb`).
"SoA" is the state-of-the-art technology of each use case (the technology name is in the hover text).
The figure shows only the use cases with LCA results and costs (Fusion plant pipe has no LCA model).

In [36]:
from l4m_lca.constants import USE_CASE_ORDER
from l4m_lca.costs import read_production_costs

production_costs = read_production_costs()
ghg_cost_fig = plots.ghg_cost_bars(workflow_df, production_costs, DEFAULT_IMPACT, STAGE_TITLES,
                                   USE_CASE_ORDER, stages=("pre", "join", "post"))

display(ghg_cost_fig)
# Image for a report (needs the kaleido package):
# plots.for_print(ghg_cost_fig, width=1800).write_image(f"{BG.results}/ghg_production_costs.png", scale=2)

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


FigureWidget({
    'data': [{'customdata': array([['0.7257685442567728', 'standard laser'],
                                   ['1.3036049921739146', 'L4M-solution']], dtype=object),
              'hovertemplate': ('<b>Heat exchanger, %{customdat' ... ':.1f}% of total<extra></extra>'),
              'legendgroup': 'pre:alkaline_degreasing',
              'legendrank': 2,
              'marker': {'color': '#2a78d6',
                         'line': {'color': 'white', 'width': 1.5},
                         'pattern': {'fgcolor': 'white', 'fgopacity': 0.75, 'shape': '', 'size': 9, 'solidity': 0.3}},
              'name': 'Alkaline degreasing',
              'showlegend': True,
              'type': 'bar',
              'uid': '0f8db8c4-0a35-4181-b73d-1ac67d9dda6a',
              'x': [SoA, L4M-solution],
              'xaxis': 'x',
              'y': {'bdata': '9/8ThFzJxD/3/xOEXMnEPw==', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': array([['0.608289983426401',